# 05 — Web Search RAG

**Time‑sensitive** queries (news, live events, recent developments) need *fresh* data that isn't in any pre‑indexed document store. The `WebSearchStrategy` uses a `WebRetriever` (which wraps DuckDuckGo) to fetch live results.

```
Query (time‑sensitive)
    │
    ▼
WebRetriever (DuckDuckGo / Google)
    │
    ▼
WebSearchStrategy → LLM → Answer + Sources
```

## 1. Imports

In [ ]:
import sys
sys.path.insert(0, "..")

try:
    from backend.adaptive_rag.retrievers.web_retriever import WebRetriever
    from backend.adaptive_rag.strategies.web_search_strategy import WebSearchStrategy
    print("✓ Imported WebRetriever and WebSearchStrategy")
except ImportError as e:
    print(f"⚠ Backend import: {e}")
    print("→ Understanding web search flow with mock data")

## 2. How WebRetriever Works

`WebRetriever` uses `duckduckgo_search` to perform a live search and formats results into the same dict structure used throughout the pipeline:

In [ ]:
try:
    retriever = WebRetriever()
    print("✓ WebRetriever instantiated (DuckDuckGo backend)")
    print("  Note: first call may be slow due to network request")
except Exception as e:
    print(f"⚠ Could not instantiate WebRetriever: {e}")
    print("→ Using mock retriever for demonstration")

    class WebRetriever:
        async def retrieve(self, query, top_k=5):
            return [
                {"content": f"Mock result about {query}. This simulates a web search result.",
                 "source": f"https://example.com/result-{i}",
                 "title": f"Result {i} for {query}",
                 "relevance_score": 0.85}
                for i in range(top_k)
            ]
    retriever = WebRetriever()

## 3. Demo: Web Search

In [ ]:
import asyncio

async def show_web_search(query):
    try:
        results = await retriever.retrieve(query, top_k=3)
        print(f"Query: {query}\n")
        print(f"Found {len(results)} results:\n")
        for i, r in enumerate(results, 1):
            print(f"  [{i}] {r.get('title', 'No title')}")
            print(f"      Source: {r['source']}")
            print(f"      Content: {r['content'][:150]}...")
            print()
    except Exception as e:
        print(f"Search failed (expected if offline): {e}")

asyncio.run(show_web_search("latest developments in AI regulation 2025"))

## 4. WebSearchStrategy

The strategy wraps the retriever + an LLM, building a context from web results and generating an answer:

In [ ]:
class MockLLM:
    async def generate_with_context(self, query, context):
        return f"Based on web search results: {context[:100]}..."

llm = MockLLM()
strategy = WebSearchStrategy(retriever, llm)
print(f"✓ Created {strategy.name()}: {strategy.description()}")

## 5. Time‑Sensitive Query Example

Let's simulate the flow for a query the classifier would mark as `is_time_sensitive`:

In [ ]:
async def demo_web_strategy(query):
    result = await strategy.execute(query, top_k=3)
    print(f"Strategy   : {result['strategy']}")
    print(f"Query      : {result['query']}")
    print(f"Answer     : {result['answer'][:200]}")
    print(f"Web sources: {len(result['sources'])}")
    for s in result['sources']:
        print(f"            - {s['source']}")

asyncio.run(demo_web_strategy("What is the latest news about quantum computing?"))

## 6. When Web Search Shines vs. Fails

| Scenario | Web Search | Document RAG |
|----------|-----------|---------------|
| "Latest stock price of Apple" | ✅ Fresh data | ❌ Stale data |
| "Scientific consensus on climate change" | ⚠ Can be noisy | ✅ Curated papers |
| "History of the Roman Empire" | ✅ Works but shallow | ✅ Deeper sources |
| "Current weather in Tokyo" | ✅ Perfect | ❌ Not available |

> **Next:** [06 — Hybrid RAG](./06_Hybrid_RAG.ipynb) — combining documents + web